# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is published in Croissant format, which enables standardized, schema-driven loading and exploration.

### Dataset Source
Dataset Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Review available `RecordSet` and field `@id`s. Let's list all record sets and their schema to inform further analysis.

In [ ]:
# List record sets available in the dataset
record_sets = dataset.record_sets
print(f"Record sets in the dataset (by @id):\n")
for rs in record_sets:
    print(f"@id: {rs.id}; name: {rs.name}")

# For the main data table, print fields/columns and their @id values
if record_sets:
    main_record_set = record_sets[0]  # For this dataset, there is likely only one main RecordSet
    print(f"\nFields in RecordSet '{main_record_set.name}' (by @id):")
    for field in main_record_set.fields:
        col_ids = [c.id for c in field.columns] if hasattr(field, 'columns') and field.columns else []
        print(f"  Field: @id={field.id}, name={field.name}, columns={col_ids if col_ids else '[]'}")

## 3. Data Extraction

Load all records from the primary RecordSet into a pandas DataFrame. We use the exact RecordSet `@id` and refer to columns using their `@id`s as specified.

In [ ]:
# Extract and load the data from each record set into a DataFrame
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())

    # Display first few rows
    dataframes[main_record_set_id].head()


## 4. Exploratory Data Analysis (EDA)

Let's inspect numeric columns, filter records, normalize values, and group by key characteristics. This demonstration uses field/column `@id` values as references for all operations.

In [ ]:
# Identify main numeric fields for analysis
df = dataframes[main_record_set_id]

# Display columns and a sample; user may need to inspect to select a field.
print("Available columns (@id):\n", df.columns.tolist())
display(df.head())

# For demonstration, try to find a column containing 'age' (typical in clinical datasets)
from difflib import get_close_matches
likely_numeric_fields = []
for col in df.columns:
    lc = col.lower()
    if any(word in lc for word in ["age", "interval", "duration", "metastasis", "count", "number"]):
        likely_numeric_fields.append(col)

if not likely_numeric_fields:
    # If no typical candidate, fallback to first numeric dtype
    numeric_candidate = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidate):
        likely_numeric_fields = [numeric_candidate[0]]

if likely_numeric_fields:
    numeric_field_id = likely_numeric_fields[0]
    print(f"\nSelected numeric field for analysis: {numeric_field_id}")
else:
    print("[Warning] No numeric field found for demonstration.")

# Set a threshold
threshold = 50  # This can be adjusted; e.g., age > 50 if applicable
if likely_numeric_fields and numeric_field_id in df.columns:
    try:
        filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
        display(filtered_df.head())
        
        # Normalize
        field_mean = filtered_df[numeric_field_id].astype(float).mean()
        field_std = filtered_df[numeric_field_id].astype(float).std()
        norm_col = numeric_field_id + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field_id].astype(float) - field_mean) / field_std
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
    except Exception as e:
        print(f"Error with numeric operations: {e}")
else:
    print("No suitable numeric field to filter and normalize.")

# Try grouping by a common group/categorical field such as sex, anatomical location, or similar
group_candidates = [col for col in df.columns if any(w in col.lower() for w in ['sex', 'gender', 'location', 'msi', 'site'])]
if likely_numeric_fields and group_candidates:
    group_field = group_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} per {group_field} (after filter):")
        display(grouped_df.head())
else:
    print("No suitable group field found for demonstration.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib and seaborn for some basic plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

if likely_numeric_fields and numeric_field_id in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=15, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    plt.show()

# If both a numeric field and group field exist, plot grouped boxplot
if likely_numeric_fields and group_candidates:
    group_field = group_candidates[0]
    if group_field in df.columns:
        fig, ax = plt.subplots(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        ax.set_title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore a biomedical Croissant dataset with `mlcroissant`. We:

- Loaded the FAIR² colorectal dataset and obtained schema information.
- Inspected record sets and fields using their unique `@id` references.
- Loaded the main record set into a DataFrame, filtered and normalized a numeric field, and grouped data by a categorical variable.
- Visualized distributions and groupings to help surface clinical patterns.

For further work, examine the complete schema in `metadata`, review documentation for data provenance and variables, and extend visualizations to model specific hypotheses about second primary colorectal cancer.